# CityPulse AI — Chicago 311 Operations Intelligence

End-to-end walkthrough: ingest -> EDA -> SLA-risk model -> evaluation -> duplicates, hotspots, forecast -> priority queue.

**Target (working SLA proxy):** a completed request that took **more than 7 days** to close. Features are knowable at creation time only — no leakage.

In [ ]:
import json, pandas as pd, numpy as np
pd.set_option('display.max_columns', 30)
closed = pd.read_parquet('data/closed_requests.parquet')
open_df = pd.read_parquet('data/open_requests.parquet')
print(f"closed: {len(closed):,} rows | open: {len(open_df):,} rows")
closed.head()

## 1. EDA — what does Chicago ask its city services to do?

In [ ]:
closed['created_date'] = pd.to_datetime(closed['created_date'])
closed['closed_date'] = pd.to_datetime(closed['closed_date'])
closed['days_to_close'] = (closed['closed_date']-closed['created_date']).dt.total_seconds()/86400
closed = closed[(closed['days_to_close']>=0)&(closed['days_to_close']<=365)]
closed.groupby('sr_type').agg(n=('sr_number','count'), median_days=('days_to_close','median')).sort_values('n', ascending=False).head(15)

In [ ]:
closed['breach'] = (closed['days_to_close']>7).astype(int)
print('Overall breach rate (>7 days):', round(closed['breach'].mean(),3))
closed.groupby('sr_type')['breach'].agg(['mean','count']).sort_values('mean', ascending=False).head(10)

## 2. Model — XGBoost on creation-time features, temporal split

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier
for c in ['created_hour','created_day_of_week','created_month']:
    closed[c] = pd.to_numeric(closed[c], errors='coerce')
closed['duplicate_flag'] = closed['duplicate'].astype(str).str.lower().isin(['true','1']).astype(int)
CAT = ['sr_type','owner_department','origin','community_area','ward','zip_code']
NUM = ['created_hour','created_day_of_week','created_month','duplicate_flag']
for c in CAT: closed[c] = closed[c].astype(str)
closed = closed.sort_values('created_date')
cut = closed['created_date'].quantile(0.8)
train, test = closed[closed['created_date']<=cut], closed[closed['created_date']>cut].copy()
pre = ColumnTransformer([('cat', Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('oh',OneHotEncoder(handle_unknown='ignore',min_frequency=20))]), CAT),
                         ('num', Pipeline([('imp',SimpleImputer(strategy='median'))]), NUM)])
pipe = Pipeline([('pre',pre),('clf',XGBClassifier(n_estimators=350,max_depth=6,learning_rate=0.06,eval_metric='logloss',n_jobs=4,random_state=42))])
pipe.fit(train[CAT+NUM], train['breach'])
test['pred_prob'] = pipe.predict_proba(test[CAT+NUM])[:,1]
print(f"train {len(train):,} | test {len(test):,} | split at {cut.date()}")

## 3. Evaluation — against honest baselines

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
def m(y,p): return {'AUROC':round(roc_auc_score(y,p),4),'AUPRC':round(average_precision_score(y,p),4),'Brier':round(brier_score_loss(y,p),4)}
base = train['breach'].mean()
type_rate = train.groupby('sr_type')['breach'].mean()
print('Model     ', m(test['breach'], test['pred_prob']))
print('Type-only ', m(test['breach'], test['sr_type'].map(type_rate).fillna(base)))
print('Dumb base ', m(test['breach'], np.full(len(test), base)))

## 4. FIFO simulation — priority order vs first-come-first-served (K = 1,000)

In [ ]:
K=1000
fifo = test.sort_values('created_date').head(K); prio = test.sort_values('pred_prob', ascending=False).head(K)
print(f"FIFO     : mean days {fifo['days_to_close'].mean():.2f}, breach share {fifo['breach'].mean():.2f}")
print(f"Priority : mean days {prio['days_to_close'].mean():.2f}, breach share {prio['breach'].mean():.2f}")

## 5. Priority queue for open requests (full logic lives in train_model.py)

In [ ]:
metrics = json.load(open('outputs/metrics.json'))
queue = pd.read_csv('outputs/prioritized_open.csv')
print({k: metrics[k] for k in ['hotspots_flagged','high_risk_open','mean_open_breach_prob']})
queue[['sr_number','sr_type','street_address','community_area','priority_score','breach_prob','reason']].head(10)

## 6. Hotspots & forecast

In [ ]:
hot = pd.read_csv('outputs/hotspots.csv'); fc = pd.read_csv('outputs/forecast.csv')
print(hot[hot['is_hotspot']].head(10)[['community_area','sr_type','count_7d','expected_7d','ratio','z_score']].to_string(index=False))
print(fc.to_string(index=False))
print('Forecast holdout:', metrics['forecast'])

---
Run the full product with `streamlit run app.py`. All numbers above come from the real cached Chicago 311 data — see README.md for limitations.